In [1]:
import os

# --- Étape 0.5 : Configuration Globale ---
# Le nom de votre dataset est défini ici
KAGGLE_DATASET_NAME = "ms-coco2014"

# --- Définition de tous les chemins ---
# Chemins d'entrée (Lecture depuis le dataset Kaggle)
INPUT_DIR = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
IMAGE_DIR = os.path.join(INPUT_DIR, 'train2014')
ANNOTATIONS_FILE = os.path.join(INPUT_DIR, 'annotations/captions_train2014.json')

# Chemins de sortie (Écriture dans le dossier de travail Kaggle)
OUTPUT_DIR = "/kaggle/working/"
CAPTIONS_FILE = os.path.join(OUTPUT_DIR, 'captions.txt')
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'features.pkl')
TOKENIZER_FILE = os.path.join(OUTPUT_DIR, 'tokenizer.pkl')
MODEL_FILE = os.path.join(OUTPUT_DIR, 'image_captioning_model.h5')
CURVES_FILE = os.path.join(OUTPUT_DIR, 'training_curves.png')
MODEL_PLOT_FILE = os.path.join(OUTPUT_DIR, 'model_rnn_decoder.png')

print(f"Chemin des images (vérification) : {IMAGE_DIR}")
print(f"Chemin des annotations (vérification) : {ANNOTATIONS_FILE}")
print(f"Chemin de sortie (vérification) : {OUTPUT_DIR}")

# Vérifions que les chemins sont corrects
if not os.path.exists(IMAGE_DIR):
    print(f"ERREUR : Le chemin des images '{IMAGE_DIR}' est incorrect. Le dossier 'train2014' existe-t-il bien dans '{INPUT_DIR}' ?")
if not os.path.exists(ANNOTATIONS_FILE):
    print(f"ERREUR : Le chemin des annotations '{ANNOTATIONS_FILE}' est incorrect.")

Chemin des images (vérification) : /kaggle/input/ms-coco2014/train2014
Chemin des annotations (vérification) : /kaggle/input/ms-coco2014/annotations/captions_train2014.json
Chemin de sortie (vérification) : /kaggle/working/


In [2]:
# --- Imports Globaux ---
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Dropout, Add
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, plot_model

import numpy as np
import pickle
import string
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt

# Vérification GPU
print("--- Vérification du GPU ---")
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print("ATTENTION : Aucun GPU n'est détecté. L'exécution sera extrêmement lente.")
else:
    print(f"GPU détecté : {gpus[0].name}")

2025-10-21 18:32:21.701219: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761071541.971108      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761071542.045564      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


--- Vérification du GPU ---
GPU détecté : /physical_device:GPU:0


In [3]:
def main_captions():
    print(f"Chargement du fichier d'annotations : {ANNOTATIONS_FILE}")
    try:
        with open(ANNOTATIONS_FILE, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"ERREUR : Fichier non trouvé '{ANNOTATIONS_FILE}'. Vérifiez votre variable 'KAGGLE_DATASET_NAME'.")
        return
        
    print("Fichier chargé. Traitement des annotations...")

    id_to_filename = {}
    for image_info in tqdm(data['images'], desc="Mappage ID->Fichier"):
        id_to_filename[image_info['id']] = image_info['file_name']
    
    count = 0
    with open(CAPTIONS_FILE, 'w', encoding='utf-8') as f_out:
        for annotation in tqdm(data['annotations'], desc="Écriture des légendes"):
            image_id = annotation['image_id']
            caption = annotation['caption'].strip().replace('\n', ' ').replace('\r', ' ')
            
            if image_id in id_to_filename:
                filename = id_to_filename[image_id]
                f_out.write(f"{filename} {caption}\n")
                count += 1
    
    print(f"\n--- Étape 1 (Légendes) Terminée. {count} légendes écrites dans {CAPTIONS_FILE} ---")

# Exécuter l'étape 1
main_captions()

Chargement du fichier d'annotations : /kaggle/input/ms-coco2014/annotations/captions_train2014.json
Fichier chargé. Traitement des annotations...


Écriture des légendes: 100%|██████████| 414113/414113 [00:00<00:00, 1207957.34it/s]



--- Étape 1 (Légendes) Terminée. 414113 légendes écrites dans /kaggle/working/captions.txt ---


In [4]:
def get_cnn_model():
    base_model = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
    cnn_model = Model(inputs=base_model.input, outputs=base_model.output, name="InceptionV3_Encoder")
    cnn_model.trainable = False
    return cnn_model

def extract_image_features(image_path, model):
    try:
        img = load_img(image_path, target_size=(299, 299))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_preprocessed = preprocess_input(img_array)
        features = model.predict(img_preprocessed, verbose=0)
        return features.flatten()
    except Exception as e:
        print(f"Erreur lors du traitement de {image_path}: {e}")
        return None

def main_preprocessing():
    print("Chargement du modèle InceptionV3...")
    model = get_cnn_model()
    print("Modèle chargé.")

    try:
        all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')]
    except FileNotFoundError:
        print(f"ERREUR : Dossier images non trouvé '{IMAGE_DIR}'. Vérifiez votre variable 'KAGGLE_DATASET_NAME'.")
        return
        
    print(f"Total d'images trouvées : {len(all_images)}")
    if len(all_images) == 0:
        print("ERREUR : 0 images trouvées. Le chemin est probablement incorrect.")
        return

    features_dict = {}
    
    # La vitesse ici devrait être > 100 it/s
    for image_name in tqdm(all_images, desc="Extraction (Vitesse Kaggle)"):
        image_path = os.path.join(IMAGE_DIR, image_name)
        features = extract_image_features(image_path, model)
        if features is not None:
            features_dict[image_name] = features

    print(f"\nExtraction terminée. {len(features_dict)} features extraites.")
    print(f"Sauvegarde dans {FEATURES_FILE}...")
    with open(FEATURES_FILE, 'wb') as f:
        pickle.dump(features_dict, f)
    
    print(f"\n--- Étape 2 (Images) Terminée. '{FEATURES_FILE}' est prêt. ---")

# Exécuter l'étape 2
main_preprocessing()

I0000 00:00:1761071559.158222      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761071559.158977      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Chargement du modèle InceptionV3...
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Modèle chargé.
Total d'images trouvées : 82783


Extraction (Vitesse Kaggle):   0%|          | 0/82783 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1761071567.081917      63 service.cc:148] XLA service 0x7c0344003750 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761071567.082794      63 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761071567.082814      63 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761071568.046246      63 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1761071572.623578      63 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Extraction (Vitesse Kaggle): 100%|██████████| 82783/82783 [2:26:55<00:00,  9.39it/s]



Extraction terminée. 82783 features extraites.
Sauvegarde dans /kaggle/working/features.pkl...

--- Étape 2 (Images) Terminée. '/kaggle/working/features.pkl' est prêt. ---


In [5]:
# --- 1. Fonctions de Chargement et Préparation ---

def load_captions(filename):
    captions_dict = {}
    with open(filename, 'r') as f:
        for line in f:
            tokens = line.split()
            if len(line) < 2: continue
            image_id, image_caption = tokens[0], tokens[1:]
            image_caption = ' '.join(image_caption)
            if image_id not in captions_dict:
                captions_dict[image_id] = []
            captions_dict[image_id].append(image_caption)
    return captions_dict

def clean_captions(captions_dict):
    table = str.maketrans('', '', string.punctuation)
    for key, caption_list in captions_dict.items():
        for i in range(len(caption_list)):
            caption = caption_list[i]
            caption = caption.split()
            caption = [word.lower() for word in caption]
            caption = [w.translate(table) for w in caption]
            caption = [word for word in caption if len(word) > 1]
            caption = [word for word in caption if word.isalpha()]
            caption_list[i] = '<start> ' + ' '.join(caption) + ' <end>'
    return captions_dict

def load_image_features(filename):
    with open(filename, 'rb') as f:
        features = pickle.load(f)
    return features

def to_vocabulary(captions_dict):
    all_captions = set()
    for key in captions_dict.keys():
        [all_captions.add(d) for d in captions_dict[key]]
    return list(all_captions)

def create_tokenizer(captions_list):
    tokenizer = Tokenizer(num_words=10000, oov_token="<unk>")
    tokenizer.fit_on_texts(captions_list)
    vocab_size = len(tokenizer.word_index) + 1
    return tokenizer, vocab_size

def get_max_length(captions_dict):
    all_captions = to_vocabulary(captions_dict)
    return max(len(d.split()) for d in all_captions)

# --- 2. Générateur de Données ---

def data_generator(captions_dict, features_dict, tokenizer, max_length, vocab_size, batch_size):
    X1_batch, X2_batch, y_batch = [], [], []
    n = 0
    while True:
        for image_id, caption_list in captions_dict.items():
            if image_id not in features_dict:
                continue
            image_features = features_dict[image_id]
            for caption in caption_list:
                n += 1
                sequence = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(sequence)):
                    X1_batch.append(image_features)
                    in_seq = sequence[:i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                    X2_batch.append(in_seq)
                    out_word = sequence[i]
                    out_word_one_hot = to_categorical([out_word], num_classes=vocab_size)[0]
                    y_batch.append(out_word_one_hot)
                if n == batch_size:
                    yield ([np.array(X1_batch), np.array(X2_batch)], np.array(y_batch))
                    X1_batch, X2_batch, y_batch = [], [], []
                    n = 0

# --- 3. Définition du Modèle RNN (Décodeur) ---

def build_rnn_decoder(vocab_size, max_length, embedding_dim, lstm_units, feature_dim):
    input_features = Input(shape=(feature_dim,))
    fe1 = Dropout(0.4)(input_features)
    fe2 = Dense(256, activation='relu')(fe1)

    input_sequence = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(input_sequence)
    se2 = Dropout(0.4)(se1)
    se3 = LSTM(256)(se2)

    decoder1 = Add()([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)
    
    model = Model(inputs=[input_features, input_sequence], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model
